# Session 9 — Understanding Uncertainty: Sandbox

> "Every measure, without knowledge of its uncertainty, is meaningless."

This notebook is the code companion to the visual [AI Evaluation Playgrounds](../../playground/index.html). Each section here lets you **run** the experiment that the playground lets you **see** — and modify it.

We use synthetic scoring data (no API calls) so you can iterate fast. The shape of the uncertainty transfers directly to real LLM-judge pipelines.

## What this notebook covers

| Part | Source of uncertainty | Playground link |
|------|----------------------|-----------------|
| 1 | Sampling noise (same eval, different numbers) | [sampling_noise](../../playground/sampling_noise.html) |
| 2 | Sampling bias (test set ≠ production) | [sampling_bias](../../playground/sampling_bias.html) |
| 3 | Multiple hypothesis testing (the "winner" got lucky) | [multiple_hypothesis_testing](../../playground/multiple_hypothesis_testing.html) |
| 4 | Judge noise and compounding | [judges_and_compounding](../../playground/judges_and_compounding.html) |

Each part is a short experiment: **simulate → observe → quantify → mitigate**.

Run cells in order. Change parameters. See what breaks.

## 0. Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(42)  # deterministic: change the seed to see a different run
print("ready")

---
## Part 1 — Sampling noise

**Claim:** if you evaluate the same system twice on different samples, you get different numbers.

**Setup:** imagine a judge scoring answers on a 1–5 rubric. The system's **true** population mean is fixed. We observe it through a finite sample of size `n`. What does the observed mean look like?

In [ ]:
TRUE_MEAN = 3.8         # the system's true mean score on a 1–5 scale (we'd never know this in real life)
TRUE_STD  = 0.9         # per-item score noise
N         = 30          # sample size — try 10, 30, 100, 500
N_RUNS    = 200         # how many times we repeat the eval (to see the wobble)

def score_sample(n):
    """Simulate n judged scores, clipped to [1, 5]."""
    raw = rng.normal(TRUE_MEAN, TRUE_STD, size=n)
    return np.clip(raw, 1, 5)

means = np.array([score_sample(N).mean() for _ in range(N_RUNS)])
print(f"n={N}: observed mean across {N_RUNS} runs — min={means.min():.2f}, max={means.max():.2f}, spread={means.max()-means.min():.2f}")
print(f"True mean is {TRUE_MEAN:.2f}; the observed mean wobbles ±{np.std(means):.2f}")

fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(means, bins=25, edgecolor='black')
ax.axvline(TRUE_MEAN, color='red', linestyle='--', label=f'true mean = {TRUE_MEAN}')
ax.set_xlabel(f'Observed mean from n={N} samples'); ax.set_ylabel('Frequency'); ax.legend()
plt.tight_layout(); plt.show()

### 1a. Quantifying it: confidence interval via bootstrap

Given **one** sample of size `n`, what can we say about the true mean? Bootstrap the sample and report a 95% CI.

In [ ]:
sample = score_sample(N)

B = 10_000
boot_means = np.array([rng.choice(sample, size=N, replace=True).mean() for _ in range(B)])
ci_low, ci_high = np.percentile(boot_means, [2.5, 97.5])

print(f"Observed sample mean: {sample.mean():.3f}")
print(f"95% bootstrap CI:     [{ci_low:.3f}, {ci_high:.3f}]  (width = {ci_high - ci_low:.3f})")
print(f"True mean was:         {TRUE_MEAN:.3f}   — does the CI contain it?")

### 1b. Beta posterior for "is the score ≥ threshold?"

Often we don't care about the raw mean — we care about *"what fraction of answers pass?"* Binarize (score ≥ 4 = pass) and compute a Beta posterior.

In [ ]:
THRESHOLD = 4.0
passes = (sample >= THRESHOLD).sum()
fails  = N - passes
print(f"Sample: {passes}/{N} pass ({passes/N:.1%})")

# Beta posterior with a uniform prior: Beta(passes + 1, fails + 1)
posterior = stats.beta(passes + 1, fails + 1)
cred_low, cred_high = posterior.ppf([0.025, 0.975])
print(f"95% credible interval on pass rate: [{cred_low:.1%}, {cred_high:.1%}]")
print(f"Width: {(cred_high - cred_low)*100:.1f} percentage points")

# Sanity: try N=10 vs N=500 above and re-run — width shrinks as ~1/sqrt(N).

> **Try this:** change `N` to `10`, `100`, and `500`. How much does the CI width change? At what `N` would you feel comfortable shipping based on the pass rate?

---
## Part 2 — Sampling bias: when your test set lies

More data doesn't help here — it just gets you wrong more confidently.

**Setup:** an English-heavy test set (90% English, 10% Spanish), but the production traffic is actually 40%/60%. The system performs very differently in each language.

In [ ]:
# True per-language means
mean_en, std_en = 4.3, 0.6
mean_es, std_es = 3.1, 0.8

def sample_mixed(n, frac_en):
    n_en = rng.binomial(n, frac_en)
    en = rng.normal(mean_en, std_en, size=n_en)
    es = rng.normal(mean_es, std_es, size=n - n_en)
    return np.clip(np.concatenate([en, es]), 1, 5)

N = 1000  # lots of data

test_set  = sample_mixed(N, frac_en=0.90)
production = sample_mixed(N, frac_en=0.40)

print(f"Test-set mean   (90% English):  {test_set.mean():.3f}")
print(f"Production mean (40% English):  {production.mean():.3f}")
print(f"Gap: {test_set.mean() - production.mean():+.3f}")
print()
print("Collecting 10x more biased data would NOT close this gap — it would just sharpen the illusion.")

> **Try this:** crank `N` up to 100,000. The test-set mean becomes extremely precise — but it still converges to the *wrong* answer. More data doesn't help when your sampling is biased.

---
## Part 3 — Multiple hypothesis testing

You try 10 prompt variants. The "winner" scores 4.3 on the eval. Ship it?

**Catch:** if *all 10 variants* have the same true score, and you pick the best sample mean, you're selecting on noise. The "winner" just got lucky.

In [ ]:
N_VARIANTS = 10
N_PER_VAR  = 50
TRUE_MEAN  = 4.0  # all variants are identical in truth
TRUE_STD   = 0.7
N_TRIALS   = 2000  # repeat the whole shoot-out many times

winners = np.empty(N_TRIALS)
for t in range(N_TRIALS):
    sample_means = rng.normal(TRUE_MEAN, TRUE_STD / np.sqrt(N_PER_VAR), size=N_VARIANTS)
    winners[t] = sample_means.max()

print(f"True mean of every variant:       {TRUE_MEAN:.3f}")
print(f"Mean of the 'winner's' score:     {winners.mean():.3f}")
print(f"95th percentile of winner scores: {np.percentile(winners, 95):.3f}")
print()
print(f"The naive \"best sample wins\" reports ~{winners.mean() - TRUE_MEAN:+.2f} above truth, by construction.")

fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(winners, bins=30, edgecolor='black')
ax.axvline(TRUE_MEAN, color='red', linestyle='--', label=f'true mean = {TRUE_MEAN}')
ax.set_xlabel('Winner\'s reported score'); ax.set_ylabel('Frequency'); ax.legend()
plt.tight_layout(); plt.show()

### 3a. Mitigation: held-out set

Split into dev / held-out. Pick the winner on dev, **report on held-out**.

In [ ]:
N_TRIALS = 2000
heldout_scores = np.empty(N_TRIALS)
for t in range(N_TRIALS):
    dev_means     = rng.normal(TRUE_MEAN, TRUE_STD / np.sqrt(N_PER_VAR), size=N_VARIANTS)
    heldout_means = rng.normal(TRUE_MEAN, TRUE_STD / np.sqrt(N_PER_VAR), size=N_VARIANTS)
    winner_idx = dev_means.argmax()
    heldout_scores[t] = heldout_means[winner_idx]

print(f"True mean:                             {TRUE_MEAN:.3f}")
print(f"Winner's held-out score (mean):        {heldout_scores.mean():.3f}")
print()
print(f"Held-out reporting is unbiased: {heldout_scores.mean() - TRUE_MEAN:+.3f} from truth.")
print("This is why you ALWAYS keep a held-out set when selecting.")

> **Try this:** increase `N_VARIANTS` to 100. What happens to the naive-winner mean? Does held-out still fix it?

---
## Part 4 — Judge noise and compounding

The judge is itself noisy. Every uncertainty source from above **re-applies** at the judge level. They compound.

In [ ]:
# The system's true score per item
SYSTEM_MEAN = 3.8
SYSTEM_STD  = 0.9

# Judge noise: each judge sample adds its own noise
JUDGE_STD = 0.5  # set to 0 to see the no-compounding baseline

N = 100

def score_item():
    true = rng.normal(SYSTEM_MEAN, SYSTEM_STD)
    observed = true + rng.normal(0, JUDGE_STD)
    return np.clip(observed, 1, 5)

N_RUNS = 1000
no_judge_noise = []
with_judge_noise = []
for _ in range(N_RUNS):
    pure   = np.clip(rng.normal(SYSTEM_MEAN, SYSTEM_STD,  size=N), 1, 5).mean()
    judged = np.array([score_item() for _ in range(N)]).mean()
    no_judge_noise.append(pure)
    with_judge_noise.append(judged)

print(f"Without judge noise: observed mean spread = {np.std(no_judge_noise):.3f}")
print(f"With judge noise:    observed mean spread = {np.std(with_judge_noise):.3f}")
print()
print("The CI gets wider. If you treat your judge as ground truth, your uncertainty is understated.")

> **Try this:** raise `JUDGE_STD` to 1.0. How much does the reported spread change? At what point does the judge noise dominate the system noise?

---
## Wrap-up

Every measurement has uncertainty. The question isn't *whether* — it's *how much*, and *does it matter for the decision I'm about to make*.

**Three takeaways:**

1. **Quantify first.** A number without a CI is a guess wearing a costume.
2. **Some uncertainty shrinks with more data. Some doesn't.** Sampling bias, hypothesis-testing winner's curse, and temporal drift do not go away with a bigger n.
3. **Decisions, not numbers, are the unit.** A wide CI is fine if it still lets you decide. A narrow CI is useless if it's centred on the wrong population.

## Going further

- [Bootstrap Lab](../../playground/bootstrap_lab.html) — hands-on practice
- [Distribution Explorer](../../playground/distribution_explorer.html) — visual intuition for the distributions you saw in code
- [Full playground index](../../playground/index.html) — all 9 sources of uncertainty with visual demos
- Session 10 notebook — **Designing Reliable Experiments** — will extend the methods here into experimental design and honest reporting